# 03 - Attribution patching

*Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2*

Notebook 02 measured head importance the expensive way, running the model once for every head. This notebook produces the same kind of score using one forward pass and one backward pass for all 144 heads at once.

Attribution patching is a first-order approximation of activation patching (Nanda, 2023). It estimates the effect of swapping an activation instead of carrying out the swap. That means agreement with notebook 02 is partly expected - the point of the study is to find out how good the approximation is and where it breaks down.

It stands in here for the LM Transparency Tool (Tufanov et al., 2024; Ferrando and Voita, 2024), which belongs to the same family of efficient gradient-based methods and produces a comparable per-head score.

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
print("GPT-2 small:", n_layers, "layers x", n_heads, "heads | device:", device)

## The same dataset as notebook 02

Identical construction and the same random seed, so the two sets of scores can be compared directly.

In [ ]:
templates = [
    "When{A} and{B} went to the shop,{S} gave a drink to",
    "When{A} and{B} got to the office,{S} sent a letter to",
]

candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]

names = []
for name in candidate_names:
    if model.to_tokens(name, prepend_bos=False).shape[1] == 1:
        names.append(name)

all_pairs = []
for name_a in names:
    for name_b in names:
        if name_a != name_b:
            all_pairs.append((name_a, name_b))

random.seed(0)
random.shuffle(all_pairs)
pairs = all_pairs[:50]

clean_prompts = []
corrupted_prompts = []
io_tokens = []
s_tokens = []

for name_a, name_b in pairs:
    for template in templates:
        clean_prompts.append(template.format(A=name_a, B=name_b, S=name_b))
        corrupted_prompts.append(template.format(A=name_a, B=name_b, S=name_a))
        io_tokens.append(model.to_single_token(name_a))
        s_tokens.append(model.to_single_token(name_b))

clean_tokens = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)
N = len(clean_prompts)
print("sentences:", N)

## Metric and baselines

Two versions of the metric are needed here.

The first returns an ordinary number and is used for reporting. The second keeps the result as a tensor, which is what allows the gradient to be calculated from it. The maths is the same in both cases.

In [ ]:
def mean_logit_difference_tensor(logits):
    """Same calculation, but the result stays a tensor so it can be differentiated."""
    final_logits = logits[:, -1, :]

    total = final_logits[0, io_tokens[0]] - final_logits[0, s_tokens[0]]
    for i in range(1, N):
        total = total + final_logits[i, io_tokens[i]] - final_logits[i, s_tokens[i]]

    return total / N

def mean_logit_difference(logits):
    return mean_logit_difference_tensor(logits).item()

In [ ]:
with torch.no_grad():
    CLEAN_BASELINE = mean_logit_difference(model(clean_tokens))
    CORRUPTED_BASELINE = mean_logit_difference(model(corrupted_tokens))

SCALE = CLEAN_BASELINE - CORRUPTED_BASELINE

print("clean baseline:    ", round(CLEAN_BASELINE, 3))
print("corrupted baseline:", round(CORRUPTED_BASELINE, 3))
print("scale:             ", round(SCALE, 3))

## Collecting activations and gradients

Three things are needed for each head:

1. its output on the clean sentence
2. its output on the corrupted sentence
3. the gradient of the logit difference with respect to that output

Forward hooks collect the activations and backward hooks collect the gradients. Both are plain functions that write into a dictionary.

No timing is recorded here. The first backward pass in a session includes one-off setup costs, so it is not a fair measurement - notebook 04 times both methods properly, using warm-up passes and GPU synchronisation.

In [ ]:
# clean activations, no gradients needed
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)

corrupted_activations = {}
corrupted_gradients = {}

def save_activation(activation, hook):
    corrupted_activations[hook.name] = activation.detach()

def save_gradient(gradient, hook):
    corrupted_gradients[hook.name] = gradient.detach()

In [ ]:
model.reset_hooks()
for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    model.add_hook(hook_name, save_activation, "fwd")
    model.add_hook(hook_name, save_gradient, "bwd")

torch.set_grad_enabled(True)
mean_logit_difference_tensor(model(corrupted_tokens)).backward()
torch.set_grad_enabled(False)
model.reset_hooks()

print("activations and gradients collected for all", n_layers, "layers")

## Estimating importance

For each head the estimate is:

`(clean activation - corrupted activation) x gradient`

added up over the positions, the head dimensions and the sentences. In words, it is how far the head's activity would move if it were made clean, multiplied by how much the answer cares about that activity.

Dividing by the same scale used in notebook 02 puts both sets of scores on the same footing.

In [ ]:
attribution_scores = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    difference = clean_cache[hook_name] - corrupted_activations[hook_name]
    contribution = difference * corrupted_gradients[hook_name]

    for head in range(n_heads):
        # sum over sentences, positions and head dimensions
        total = contribution[:, :, head, :].sum().item()
        attribution_scores[layer, head] = total / SCALE

print("attribution scores calculated for all", n_layers * n_heads, "heads")

In [ ]:
largest = abs(attribution_scores).max()

plt.figure(figsize=(8, 6))
plt.imshow(attribution_scores, cmap="RdBu", vmin=-largest, vmax=largest)
plt.colorbar(label="estimated importance")
plt.xlabel("head")
plt.ylabel("layer")
plt.title("Attribution patching importance per head")
plt.xticks(range(n_heads))
plt.yticks(range(n_layers))
plt.show()

scored_heads = []
for layer in range(n_layers):
    for head in range(n_heads):
        scored_heads.append((attribution_scores[layer, head], layer, head))

scored_heads.sort(reverse=True)

print("top 10 heads by attribution score")
for score, layer, head in scored_heads[:10]:
    print("  layer", layer, "head", head, "->", round(score, 3))

## Result

The same regions light up as in notebook 02, and the exact numbers differ because this is an estimate rather than a measurement. How closely the two agree is the subject of notebook 04.

The cost difference is structural rather than incidental. Activation patching needs one forward pass for every head examined, so 144 in total. Attribution patching needs one forward pass and one backward pass no matter how many heads there are.

### References

- Ferrando, J. and Voita, E. (2024) *Information Flow Routes: Automatically Interpreting Language Models at Scale*.
- Nanda, N. (2023) *Attribution Patching: Activation Patching at Industrial Scale*.
- Nanda, N. and Bloom, J. (2022) *TransformerLens*.
- Tufanov, I. et al. (2024) *LM Transparency Tool: Interactive Tool for Analyzing Transformer Language Models*.